In [3]:
!pip install gensim

import numpy as np
import string
import nltk
from nltk.corpus import stopwords
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
from tabulate import tabulate
from collections import Counter


nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    filtered_tokens = [word for word in tokens if word not in stop_words]
    return " ".join(filtered_tokens)

dataset = [
    "I love playing football on the weekends",
    "I enjoy hiking and camping in the mountains",
    "I like to read books and watch movies",
    "I prefer playing video games over sports",
    "I love listening to music and going to concerts"
]


clean_dataset = [preprocess_text(doc) for doc in dataset]
print("Preprocessed Dataset:", clean_dataset)
print("-" * 50)

print("\n--- TF-IDF CLUSTERING (PREPROCESSED) ---")
vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(clean_dataset)

k = 2
km_tfidf = KMeans(n_clusters=k, random_state=42)
y_pred_tfidf = km_tfidf.fit_predict(X_tfidf)

table_data_tfidf = [["Document", "Cleaned Text", "Predicted Cluster"]]
table_data_tfidf.extend([[doc, clean, cluster] for doc, clean, cluster in zip(dataset, clean_dataset, y_pred_tfidf)])
print(tabulate(table_data_tfidf, headers="firstrow"))


cluster_label_counts_tfidf = [Counter(y_pred_tfidf)]
purity_tfidf = sum(max(cluster.values()) for cluster in cluster_label_counts_tfidf) / len(y_pred_tfidf)
print("\nTF-IDF Purity:", purity_tfidf)

print("\n--- WORD2VEC CLUSTERING (PREPROCESSED) ---")
tokenized_dataset = [doc.split() for doc in clean_dataset]

word2vec_model = Word2Vec(sentences=tokenized_dataset, vector_size=100, window=5, min_count=1, workers=4)


X_w2v = np.array([
    np.mean([word2vec_model.wv[word] for word in doc if word in word2vec_model.wv], axis=0) 
    if any(word in word2vec_model.wv for word in doc) else np.zeros(100)
    for doc in tokenized_dataset
])

km_w2v = KMeans(n_clusters=k, random_state=42)
y_pred_w2v = km_w2v.fit_predict(X_w2v)

table_data_w2v = [["Document", "Predicted Cluster"]]
table_data_w2v.extend([[doc, cluster] for doc, cluster in zip(dataset, y_pred_w2v)])
print(tabulate(table_data_w2v, headers="firstrow"))

cluster_label_counts_w2v = [Counter(y_pred_w2v)]
purity_w2v = sum(max(cluster.values()) for cluster in cluster_label_counts_w2v) / len(y_pred_w2v)
print("\nWord2Vec Purity:", purity_w2v)

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 10.9 MB/s  0:00:02eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gensim]━━━━━ 1/2 [gensim]
Preprocessed Dataset: ['love playing football weekends', 'enjoy hiking camping mountains', 'like read books watch movies', 'prefer playing video games sports', 'love listening music going concerts']
--------------------------------------------------

--- TF-IDF CLUSTERING (PREPROCESSED) ---
Document                                         Cleaned Text                           Predicted Cluster
-----------------------------------------------  -----------------------------------  -------------------
I love playing football on the weekends          love playing football weekends                         1
I enjoy hiking and camping in the mountains      enjoy hiking camping mountains                         0
I li

/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


In [4]:
import pandas as pd
import numpy as np
import string
import nltk
from nltk.corpus import stopwords
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from tabulate import tabulate

nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    filtered_tokens = [word for word in tokens if word not in stop_words]
    return " ".join(filtered_tokens)

df = pd.read_csv('customer_complaints_1 (1).csv')

complaints_text = df['text'].tolist()
clean_complaints = [preprocess_text(doc) for doc in complaints_text]

vectorizer = TfidfVectorizer(max_features=500)
X = vectorizer.fit_transform(clean_complaints)


k = 3  
km = KMeans(n_clusters=k, random_state=42)
y_pred = km.fit_predict(X)

print("\nTop terms per complaint cluster:")
order_centroids = km.cluster_centers_.argsort()[:, ::-1]
terms = vectorizer.get_feature_names_out()

for i in range(k):
    print(f"\nCluster {i}:")
    top_terms = [terms[ind] for ind in order_centroids[i, :10]]
    print(", ".join(top_terms))

print("\nSample of Clustered Complaints:")
sample_data = [["Original Text (Snippet)", "Predicted Cluster"]]
for doc, cluster in zip(complaints_text[:5], y_pred[:5]):
    # Truncating the text to 75 characters for cleaner display
    snippet = doc[:75] + "..." if len(doc) > 75 else doc
    sample_data.append([snippet, cluster])

print(tabulate(sample_data, headers="firstrow"))

FileNotFoundError: [Errno 2] No such file or directory: 'customer_complaints_1.csv'